In [ ]:
import yaml
import base64
import pandas as pd
import requests as rt
import psycopg2
import datetime
import os
import sys 

from getpass import getpass
from pandas import read_sql
from datetime import datetime, timedelta
from IPython.display import display, HTML
from yaml import load

#create yaml config 
data = {
        'version': '1.0',
        'redshift': {
            'environment': {
                'REDSHIFT_USER': 'user',
                'REDSHIFT_PASSWORD': 'pass',
                'REDSHIFT_DB': 'iris_trusted_zone',
                'REDSHIFT_HOST': 'redshift.dev.iris.informa.com',
            },
            'ports': '5439'
        },
        'query_statement': {
            'fields': 'field1,field2,field3',
            'table': 'table',
            'conditions': 'condition1=1 AND contiond2=2'
        }
}

# Writing the data to a YAML file
with open('docuR_config.yaml', 'w') as file:
    yaml.dump(data, file)

print("Data has been written to 'data.yaml'")




: 

In [ ]:
### Constants
SOURCE_DB = "source_db"
DESTINATION_DB = "destination_db"
TABLE_NAME = "table_name"
DATE_COLUMN = "date_column"
SALES_COLUMN = "sales_column"

### Functions

# connection
def connect_to_redshift_and_query(query, connection_params):
    try:
        # Establish a connection to Redshift
        conn = psycopg2.connect(
            host=connection_params['host'],
            port=connection_params['port'],
            database=connection_params['database'],
            user=connection_params['user'],
            password=connection_params['password']
        )

        # Create a cursor object to execute SQL queries
        cursor = conn.cursor()

        # Execute the SQL query
        cursor.execute(query)

        # Fetch all the rows from the query result
        result_data = cursor.fetchall()

        # Get the column names from the cursor description
        column_names = [desc[0] for desc in cursor.description]

        # Create a Pandas DataFrame from the query result
        df = pd.DataFrame(result_data, columns=column_names)

        # Close the cursor and the connection
        cursor.close()
        conn.close()

        return df

    except Exception as e:
        print("Error:", e)
        return None

# download link
def create_download_link( df, title = "Download CSV file"):  
    
    current_date = datetime.now()

    # Format the date as a string in a specific format (e.g., YYYY-MM-DD)
    formatted_date = current_date.strftime("%Y%m%d_%H%M%S")
    filename = f"reference_data_{formatted_date}.csv"
    
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

def query_format(fields ,table ,conditions ):
    sql = (f"SELECT {fields} " f"FROM {table} " f"WHERE {conditions};")
    return sql

In [ ]:
# manually input user and password from vault
user = getpass('username')
pwd = getpass('password')

#oidc_john_renacia_iiris_data_analyst_1776838235
#3PNSZiBg2Vg8Skrq1WT-

In [ ]:
# open and load config yaml
with open("docuR_config.yaml", 'r') as f:
    config = yaml.load(f,Loader=yaml.SafeLoader)
print(config['redshift']['environment']['REDSHIFT_DB'])


In [ ]:
#connection params

connection_params = {
    'database': config['redshift']['environment']['REDSHIFT_DB'], 
    'user':user,
    'password':pwd,
    'host':config['redshift']['environment']['REDSHIFT_HOST'],
    'port':config['redshift']['ports']
}

fields = config['query_statement']['fields']
table = config['query_statement']['table']
conditions = config['query_statement']['conditions']
connection_params

In [ ]:
# raw demogs query , connect 
cpna25 = """select distinct entity, source_demographic_code, source_demographic_value, iris_standard_value, iris_standard_code from iris_reference.raw_demographics where entity in ('Audience Type', 'Registration Status') and source_system = 'CDS (Convention Data Services)' and source_demographic_code in ('R','HP','D','FS','M','M1DAY','S','S1DAY','I','I1DAY','BS','BS1DAY','HB','GU','PR','FR','SM','SM1','POD','MS','MSALE','EXPC','EX','EXCA','NEXS','SPK','SPKGU','STF','VND','SH','SHG','AMBERA','AMBERD','AMBERP','BLACKLIST','IX','MM','CX','CXEX','CXMM','PENDWT','PROBPAY','PROB','PCX','DU','MEMPROB','CHGBK','XX','DPR','PPR','BSP','DP','FRP','FSP','HBP','HPP','IP','MEP','MP','MSALEP','MSP','PODP','RP','BSD','DD','DPR','FRD','FSD','HBD','HPD','ID','MD','MED','MSALED','MSD','PODD','RD','SM1D','SM1P','SMD','SMP') order by 2,3,5;"""

query_format(fields,table,conditions)
cntQry = connect_to_redshift_and_query(cpna25, connection_params)


In [ ]:
import os

# File path
a = "Reg_status_aud_type_validation.xlsx"

# Check if the file exists and is a file
if os.path.isfile(a):
    print("File exists and is a file")
else:
    print("File does not exist or is not a file")
    cntQry.to_excel("Reg_status_aud_type_validation.xlsx",sheet_name="Raw_demographics_raw_Data", index=False)

df = pd.read_excel(a)
df['concat(B,C,D)'] = df['source_demographic_code'] + df['source_demographic_value'] + df['iris_standard_value']
df.to_excel(a, index=False)

print("New column added successfully!")
df